# Gemma-12B — Climate pilot, paired A/B RE-RUN on the labelled chart

**Why this exists.** This model's climate paired result was produced on **18 August at ~19:00**,
before the chart was relabelled. The labels were added to the generator at 19:51 (`99b44a6c6`) and
the images regenerated at 20:34 (`60cbbe7e8`), so the four models run on 19 August saw a different
stimulus: every data point carries its printed value, while this model had to read the lines by eye.

Verified by blob hash on the actual paired stimuli, not inferred from timestamps —
`PNGs/metrics/realistic/1000/001_remy_ashford_c.png` is `5818d70c` at the commit that added this
model's results and `deb04154` now.

Section 4.10 puts all six models in one table. Until this re-run, two of them were on a different
chart from the other four, unstated. For the two models where the effect *can* be measured,
relabelling moved the tied-engagement figure by 5.1 points (Qwen3-VL-8B) and 22.3 points
(Ministral-3-14B), so it is not a rounding difference.

**Scope: the `metrics` paired grid only.** 25 posts × 49 cells = 1,225 trials. The single-image and
diagnostic runs in the original notebook are not repeated — Section 4.10's diagnostics were about
the baseline response and already have their own labelled re-run.

The old results are **moved, not deleted**, to `outputs/pre_label_backup/`, matching what already
exists for Qwen3-VL-8B and Ministral-3-14B.


In [ ]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "tokenizers>=0.22.0,<=0.23.0",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

Restart the kernel after running the setup cell above.


In [ ]:
!nvidia-smi

In [ ]:
# --- HF Auth ---
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

# --- Path setup ---
from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

# --- Load Gemma model ---
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-12B-it"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
device = model.device

In [ ]:
from e1_utils.inference_gemma import run_inference_gemma

## Configuration — the same 25-image sample as every other climate-pilot model


In [ ]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: climate pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_climate/  -- shared across all 4 climate-pilot models, so they see the identical 25-image sample
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "climate_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2)
correct_base = ROOT_DIR / "climate_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs/metrics/realistic"


## Check the stimuli are the labelled ones


In [ ]:
# The stimuli must be the labelled ones. Assert it rather than trust the working copy.
import subprocess
LABEL_COMMIT = "60cbbe7e8"          # "html to png line chart with labels"
r = subprocess.run(["git", "-C", str(ROOT_DIR), "merge-base", "--is-ancestor", LABEL_COMMIT, "HEAD"],
                   capture_output=True)
assert r.returncode == 0, "This checkout predates the labelled stimuli — git pull first."

probe = correct_base / "1000" / f"{selected_numbers[0]}_remy_ashford_c.png"
assert probe.exists(), f"missing stimulus: {probe}"
print(f"✅ labelled stimuli present\n   {probe}\n   {probe.stat().st_size:,} bytes")


In [ ]:
# Look at one before spending 1,225 trials — the values should be printed on every point.
from IPython.display import Image, display
display(Image(filename=str(correct_base / "1000" / f"{selected_numbers[0]}_remy_ashford_c.png")))


## Move the pre-labelling results aside

The runner resumes from an existing output file, so this must happen before the run.


In [ ]:
# Move the pre-labelling results aside. This is REQUIRED, not tidiness:
# run_e1_metrics_paired resumes from an existing output file, so if the old JSON is still in
# place the run below would skip all 1,225 trials and change nothing.
import shutil

BACKUP = OUTPUT_DIR / "pre_label_backup"
BACKUP.mkdir(parents=True, exist_ok=True)

for name in ("e1_results_metrics_paired.json", "e1_analysis_metrics_paired.csv"):
    s, d = OUTPUT_DIR / name, BACKUP / name
    if d.exists() and s.exists():
        print(f"⚠ {name}: a backup already exists AND the file is still in outputs/ — "
              f"inspect manually, not overwriting either.")
    elif d.exists():
        print(f"⏭ {name}: already backed up")
    elif s.exists():
        shutil.move(str(s), str(d))
        print(f"📦 moved {name} → pre_label_backup/")
    else:
        print(f"·  {name}: not present")

assert not (OUTPUT_DIR / "e1_results_metrics_paired.json").exists(), \
    "old results still in outputs/ — the run would resume from them instead of starting fresh"
print("\n✅ clear to run")


## Re-run the paired grid (1,225 trials)


In [ ]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_gemma)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

## Results


In [ ]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")

## Old vs. new


In [ ]:
# Old vs new, on the one number Section 4.10 reports.
import json

def diagonal_pct(path):
    d = [r for r in json.loads(Path(path).read_text()) if r["answer"] in ("A", "B")]
    t = [r for r in d if r["correct_scale"] == r["incorrect_scale"]]
    a = sum(1 for r in t if r["answer"] == "A") / len(t) * 100
    return sum(1 for r in t if r["liked_variant"] == "correct") / len(t) * 100, a, len(d)

new = OUTPUT_DIR / "e1_results_metrics_paired.json"
old = OUTPUT_DIR / "pre_label_backup" / "e1_results_metrics_paired.json"

n_pct, n_a, n_n = diagonal_pct(new)
print(f"labelled chart (new)   tied-engagement {n_pct:5.1f}%   A-rate {n_a:5.1f}%   n={n_n}")
if old.exists():
    o_pct, o_a, o_n = diagonal_pct(old)
    print(f"unlabelled chart (old) tied-engagement {o_pct:5.1f}%   A-rate {o_a:5.1f}%   n={o_n}")
    print(f"\nchange: {n_pct - o_pct:+.1f} points")
    print("\nA tied-engagement figure only means competence if the A-rate is near 50%.")
    print("Both runs answering one slot on 80%+ of tied trials means neither measures competence,")
    print("and the change between them is not a change in ability.")
else:
    print("(no pre-label backup found for comparison)")
